# Step 6 — LambdaRank re-ranker and the final comparison

Combines the hand-crafted features with the cross-fitted tower score in a LightGBM LambdaRank model and compares every model on the test users.

### Setup (run first)

**On Google Colab:** *Runtime → Change runtime type → GPU* (T4 is enough; A100/L4 is faster). Put the two raw JDsearch
files in `MyDrive/bt4222/raw/` (`user_behavior_data.txt`, `product_meta_data.txt`). The cell mounts Drive, clones the
repository, installs the requirements and writes every output to `MyDrive/bt4222/outputs/`, so results persist between
sessions and the notebooks can be run one after another (or on different days).

**Locally:** run the notebook from the repository with the virtual environment active; data is read from `data/raw/`
and outputs go to `outputs/`.

`CONFIG = "configs/smoke.yaml"` runs everything on a 2,000-user sample with tiny models (minutes, CPU) to check the code;
`configs/default.yaml` is the real run. Every stage skips work whose outputs already exist; pass `"--force"` to redo one.

In [ ]:
import os, subprocess, sys
from pathlib import Path

CONFIG = "configs/default.yaml"          # "configs/smoke.yaml" for a quick check
REPO_URL = "https://github.com/harvestingmoon/bt4222-two-tower-recsys.git"
DRIVE_DIR = "/content/drive/MyDrive/bt4222"   # Colab: raw/ (input) and outputs/ (written by the pipeline)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/bt4222-two-tower-recsys")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from jds.notebook import Runner, banner
run = Runner(CONFIG, drive_dir=DRIVE_DIR if IN_COLAB else None)

## 6.1 Re-ranker (`scripts/05_train_ranker.py`, `src/jds/ranker/`)

LightGBM `lambdarank`, label gain [0, 1, 3, 7] (= 2^label − 1), trained on training users' slates, early-stopped on
validation users. Inputs: the step-2 features + tower score, its within-slate percentile and gap to the slate's best.
Needs ~15 GB RAM at full scale: on a standard Colab runtime add `"--set", "ranker.train_user_frac=0.5"`.

In [ ]:
run.stage("05_train_ranker.py")

In [ ]:
run.stage("06_predict.py")

## 6.2 Results

**Sample splits:** users are split 80/10/10 (train / validation / test); only users with at least one positive are
evaluated. Training-set numbers are on a 25% subsample of training users and use out-of-fold CTR and tower scores, so
they are comparable with validation/test (a large train–test gap would signal overfitting).

In [ ]:
banner()
run.table("summary_test.csv", caption="Test users: every model on the full displayed slates")

In [ ]:
banner()
run.table("summary_all_splits.csv", caption="Training / validation / test performance")

In [ ]:
run.table("ranker_feature_importance.csv", highlight=()).data.head(25)